# Week 3 — Ask for a qualitative suggestion in JSON

**Research task:** Ask for a provisional theme, a source excerpt ID and a question for the researcher, then return to the cited passage.

**Python introduced:** a route selector, `if/else`, lists of dictionaries, `json.dumps(...)` and `json.loads(...)`.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christopherbarrie/GenAI_Soc2026/blob/main/workbook/session03/session03_qualitative_interpretation.ipynb)

Colab supports the OpenRouter route only. Local JupyterLab or VS Code is canonical because it can also reach Ollama.

In [ ]:
# Colab setup: clone the public repository when running in Colab.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo = SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists():
        setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)], check=True)
    setup_os.chdir(setup_repo / 'workbook' / 'session03')
print('Working folder:', SetupPath.cwd())

## Load the course settings and SDKs

**Input:** installed Python packages, `config/course_models.json`, and—if it is not already set—the hidden OpenRouter key. **Operations:** `import` makes an installed tool available; `Path.cwd()` gives Python the current folder; the `while` block walks upward until it finds the course configuration; `json.loads(...)` turns the file's JSON text into a dictionary; square brackets retrieve the two model names. **Output:** `HOSTED_MODEL` and `LOCAL_MODEL` are strings. `getpass(...)` accepts the key without echoing it. The folder-search code is supplied setup and is not assessed.


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Choose one route and store two identified excerpts

`ROUTE` is a string controlling which later branch runs. `excerpts` is a list containing two dictionaries; every dictionary keeps an excerpt ID beside its text. Keeping IDs with passages makes it possible to test whether a model's cited evidence actually exists.


In [ ]:
ROUTE = "ollama"  # change to "openrouter" if preferred
excerpts = [
    {"id": "e01", "text": "After several exchanges, I attended the tenants' meeting."},
    {"id": "e02", "text": "I accepted help but did not attend political meetings."},
]
print("Route:", ROUTE)
print("First excerpt:", excerpts[0])

## Convert the excerpts into prompt text and construct messages

`json.dumps(excerpts)` converts the list of source dictionaries to text without discarding their IDs. The prompt states the three required output fields. The result is placed in the familiar one-message list. Its output is model input, not a theme or finding.


In [ ]:
prompt = (
    "Suggest one provisional theme. Return JSON with exactly theme, evidence_id, "
    "and question_for_researcher. Excerpts: " + json.dumps(excerpts)
)
messages = [{"role": "user", "content": prompt}]
print(prompt)

## Make the selected route's call without hiding either branch

`if ROUTE == "openrouter"` runs only when that comparison is true; otherwise `else` runs the Ollama code. Both complete calls remain visible. Each branch assigns returned text to the same name, `raw_json`, so the parsing step below is identical whichever route was chosen.


In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"},
        )
    raw_output = response.choices[0].message.content
else:
    response = ollama.chat(
        model=LOCAL_MODEL,
        messages=messages,
        format="json",
        options={"temperature": 0},
    )
    raw_output = response.message.content

print("Raw JSON text:", raw_output)

## Parse the JSON string and return to its cited source

`json.loads(raw_json)` parses the returned string into a Python dictionary. Square brackets retrieve the theme and cited ID. The `for` loop checks each source record until its ID matches; it then saves the actual excerpt. The output to assess is the pair `theme` plus `cited_excerpt`, not the theme alone.


In [ ]:
suggestion = json.loads(raw_output)
evidence_id = suggestion["evidence_id"]
cited_excerpt = None
for excerpt in excerpts:
    if excerpt["id"] == evidence_id:
        cited_excerpt = excerpt

print("Theme:", suggestion["theme"])
print("Cited excerpt:", cited_excerpt)
print("Question:", suggestion["question_for_researcher"])

# ONE CHANGE: change e02 to
# "The food deliveries helped, but I avoided the group because meetings felt hostile."

## Methodological check

Well-formed JSON makes the fields retrievable. The researcher must still decide whether the cited excerpt supports the theme and what contrary evidence changes it.
## Completion recording

Use one chosen route, change only e02 and rerun. Explain `ROUTE`, both `if/else` branches, `json.dumps`, the raw returned string, `json.loads` and the source lookup. Say whether the cited passage supports the proposed theme.

Explain every input and output aloud. Never show the shared key.